# 💻 Notebook do Aluno — Aula 13: LangGraph — StateGraph, nodes, conditional edges e HITL

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 13/14 — Módulo 4 · Grafos de estado em código**  
**1h40min**  
**StateGraph · MemorySaver · HITL · draw_mermaid**  
**Andaime 60%**  

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime da aula.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

---

## 🧩 Andaime da aula — complete as lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langgraph langchain-ollama langchain-community duckduckgo-search -q

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
from pydantic import BaseModel
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_community.tools import DuckDuckGoSearchRun
from IPython.display import Image, display
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]="https://ollama.com"
os.environ["OLLAMA_API_KEY"]=userdata.get("OLLAMA_API_KEY")

In [ ]:
llm=ChatOllama(model="gpt-oss:120b",temperature=0)

# LACUNA 1: Estado com TypedDict
class Estado(TypedDict):
    mensagens: Annotated[list, ___]  # usar add_messages
    resultados_busca: str
    qualidade: ___  # float
    iteracoes: ___  # int

class Qualidade(BaseModel):
    score:float; suficiente:bool

# LACUNA 2: node_buscar
def node_buscar(state:Estado)->dict:
    bruto=DuckDuckGoSearchRun().run(___)  # query = mensagens[0].content
    return {"resultados_busca":bruto[:1500],"iteracoes":state["iteracoes"]+1}

def node_avaliar(state:Estado)->dict:  # PRONTO
    q=(ChatPromptTemplate.from_template("P:{p}\nR:{r}\nScore:")
       |llm.with_structured_output(Qualidade)).invoke(
        {"p":state["mensagens"][0].content,"r":state["resultados_busca"]})
    return {"qualidade":q.score}

# LACUNA 3: node_responder
def node_responder(state:Estado)->dict:
    resp=llm.invoke([HumanMessage(
        f"P:{state['mensagens'][0].content}\nFonte:{state[___]}"
    )])
    return {"mensagens":[resp]}

# LACUNA 4: decidir_continuar (threshold 0.7, MAX 3)
def decidir_continuar(state:Estado)->str:
    if state["qualidade"]>=___ or state["iteracoes"]>=___:
        return ___  # "responder"
    return ___      # "buscar"

# LACUNA 5: montar grafo
builder=StateGraph(Estado)
builder.add_node("buscar",___);builder.add_node("avaliar",___);builder.add_node("responder",___)
builder.add_edge(START,"buscar");builder.add_edge("buscar",___)  # → avaliar
builder.add_conditional_edges("avaliar",___,{"buscar":"buscar","responder":"responder"})
builder.add_edge("responder",END)
pesquisador=builder.compile(checkpointer=MemorySaver())
try: display(Image(pesquisador.get_graph().draw_mermaid_png()))
except: print(pesquisador.get_graph().draw_mermaid())

---

## ✍️ Suas anotações

Registre aqui as observações da aula (qualidade dos resultados, comparações e conclusões do grupo).

---

## 🏋️ Exercícios da Aula 13

Prática do grafo de estado no domínio do grupo: montar o mini-grafo do Router, pausar para revisão humana com `interrupt_before`, calibrar o threshold e o MAX do loop condicional e trocar a fonte do node buscar pelo retriever do CKP02. Rode os andaimes na ordem e complete as lacunas — individual, ~10 min por exercício, direto no Colab.


### Exercício 1 — Monte o mini-grafo do Router · ★★☆ · 10 min

*Individual · Colab*

1. Complete `decidir_rota_ex` — o campo do estado que a função de roteamento lê.
2. Registre com `add_node` o nó que falta (os outros já estão prontos).
3. Complete a aresta inicial e a entrada que falta no mapa de `add_conditional_edges`.
4. Rode a célula — os 3 inputs devem terminar no nó correto e o `draw_mermaid()` mostra `classificar` com 3 saídas.

> **💡 Dica:** no mapa de `add_conditional_edges`, a chave é o RETORNO da função e o valor é o NOME do nó — `add_conditional_edges(no_origem, fn_rota, {"retorno": "destino"})`.


In [ ]:
# 👉 LACUNA: monte o mini-grafo do Router com LangGraph (requer o pip do lab: `llm` definido)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate

class EstadoEx(TypedDict):
    mensagens: Annotated[list, add_messages]
    rota: str

class RotaEx(BaseModel):
    destino: Literal["rag", "calculadora", "conversa"]

clf_ex = (ChatPromptTemplate.from_messages([
    ("system", """Você é um roteador de intenções. Classifique o input em exatamente um destino:
- rag: perguntas sobre documentos do domínio
- calculadora: cálculos numéricos
- conversa: saudações e qualquer outra coisa
Retorne apenas o JSON com o campo 'destino'."""),
    ("human", "{input}"),
]) | llm.with_structured_output(RotaEx))

def node_classificar(state: EstadoEx) -> dict:
    rota = clf_ex.invoke({"input": state["mensagens"][-1].content})
    return {"rota": rota.destino}

def node_rag(state: EstadoEx) -> dict:
    return {"mensagens": ["[RAG] resposta com base nos documentos do domínio"]}

def node_calc(state: EstadoEx) -> dict:
    return {"mensagens": ["[CALC] resultado da expressão"]}

def node_conversa(state: EstadoEx) -> dict:
    return {"mensagens": ["[CHAT] resposta amigável"]}

# 👉 LACUNA 1: a função lê o campo do estado com a decisão
def decidir_rota_ex(state: EstadoEx) -> str:
    return state[___]

builder_ex = StateGraph(EstadoEx)
builder_ex.add_node("classificar", node_classificar)
builder_ex.add_node("rag", node_rag)
builder_ex.add_node("calculadora", node_calc)
# 👉 LACUNA 2: registre o nó de conversa
builder_ex.add_node(___, node_conversa)
# 👉 LACUNA 3: aresta inicial (START → classificar)
builder_ex.add_edge(START, ___)
# 👉 LACUNA 4: complete o mapa de arestas condicionais (retorno → destino)
builder_ex.add_conditional_edges("classificar", decidir_rota_ex,
    {"rag": "rag", "calculadora": "calculadora", ___: ___})
for no in ["rag", "calculadora", "conversa"]:
    builder_ex.add_edge(no, END)

app_ex = builder_ex.compile()
for pergunta in ["Qual o prazo de garantia?", "Quanto é 12 × 0.9?", "Oi, tudo bem?"]:
    r = app_ex.invoke({"mensagens": [HumanMessage(pergunta)], "rota": ""})
    print(f"{pergunta[:28]:28s} → nó: {r['rota']:12s} | {r['mensagens'][-1].content[:40]}")


### Exercício 2 — Human-in-the-Loop: as 3 fases · ★★☆ · 10 min

*Individual · Colab*

1. Complete o `interrupt_before` — o grafo deve PAUSAR antes do nó de revisão — e o `thread_id` da config.
2. Rode a FASE 1 e complete o `get_state` da FASE 2 — `next` deve mostrar `('revisar',)`.
3. Complete a retomada da FASE 3 com o valor que continua do ponto suspenso sem novo input.
4. Bônus: descomente o bloco de EDIÇÃO para trocar o rascunho com `update_state` antes de retomar.

> **💡 Dica:** `invoke(None, config)` retoma do ponto exato; `update_state(config, {...})` edita o estado salvo. O par só funciona com checkpointer.


In [ ]:
# 👉 LACUNA: mini-grafo com pausa de revisão humana — as 3 fases do HITL (requer `llm`)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage

class EstadoHitl(TypedDict):
    mensagens: Annotated[list, add_messages]
    rascunho: str

def node_rascunhar(state: EstadoHitl) -> dict:
    resp = llm.invoke(state["mensagens"])
    return {"mensagens": [resp], "rascunho": resp.content}

def node_revisar(state: EstadoHitl) -> dict:
    revisao = llm.invoke([HumanMessage(
        f"Revise e melhore o texto abaixo, mantendo o tamanho:\n{state['rascunho']}")])
    return {"mensagens": [revisao], "rascunho": revisao.content}

builder_hitl = StateGraph(EstadoHitl)
builder_hitl.add_node("rascunhar", node_rascunhar)
builder_hitl.add_node("revisar", node_revisar)
builder_hitl.add_edge(START, "rascunhar")
builder_hitl.add_edge("rascunhar", "revisar")
builder_hitl.add_edge("revisar", END)

# 👉 LACUNA 1: pause ANTES do nó de revisão
app_hitl = builder_hitl.compile(checkpointer=MemorySaver(), interrupt_before=[___])
# 👉 LACUNA 2: identifique a thread da sessão
config_hitl = {"configurable": {"thread_id": ___}}

# FASE 1 — roda até a pausa (rascunhar executa; revisar fica suspenso)
app_hitl.invoke({"mensagens": [HumanMessage("Explique o que é um Router Chain em 2 linhas")],
                 "rascunho": ""}, config=config_hitl)

# FASE 2 — humano inspeciona o estado suspenso
# 👉 LACUNA 3: inspecione o estado da thread
snap = app_hitl.get_state(___)
print("Suspenso antes de:", snap.next)                # ('revisar',)
print("Rascunho:", snap.values["rascunho"])

# FASE 3 — humano aprova → retomar sem novo input
# 👉 LACUNA 4: valor que retoma do ponto suspenso
app_hitl.invoke(___, config=config_hitl)
print("Depois da retomada:", app_hitl.get_state(config_hitl).next)   # ()

# BÔNUS — humano EDITA antes de retomar (descomente para testar)
# config_ed = {"configurable": {"thread_id": "fixacao13-edit"}}
# app_hitl.invoke({"mensagens": [HumanMessage("Escreva um título de slide")],
#                  "rascunho": ""}, config=config_ed)
# app_hitl.update_state(config_ed, {"mensagens": [HumanMessage("versão editada pelo humano")]})
# app_hitl.invoke(None, config=config_ed)
# print("Depois da edição:", app_hitl.get_state(config_ed).next)


### Exercício 3 — Threshold e MAX do loop condicional · ★★☆ · 10 min

*Individual · Colab*

1. Complete o `THRESHOLD` (comece com `0.7`) e o `MAX_ITER` (limite que garante que o loop termina).
2. Complete os dois retornos de `decidir_continuar` — quando encerrar em `responder` e quando voltar ao loop.
3. Rode a célula com as 2 perguntas e registre iterações e score final.
4. Troque o `THRESHOLD` para `0.5` e depois `0.9` e rode de novo — documente a diferença no número de iterações.

> **💡 Dica:** threshold baixo = rápido e raso; alto = lento e mais fundamentado. Sem o `MAX`, um avaliador rigoroso trava o grafo em loop infinito.


In [ ]:
# 👉 LACUNA: parametrize o loop condicional do pesquisador (requer o pip do lab: `llm`)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
from pydantic import BaseModel
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun

class EstadoLoop(TypedDict):
    mensagens: Annotated[list, add_messages]
    resultados_busca: str
    qualidade: float
    iteracoes: int

class Qualidade(BaseModel):
    score: float; suficiente: bool

# 👉 LACUNA 1 e 2: threshold inicial (0.7) e limite do loop (MAX 3)
THRESHOLD = ___
MAX_ITER = ___

def node_buscar_l(state: EstadoLoop) -> dict:
    bruto = DuckDuckGoSearchRun().run(state["mensagens"][0].content)
    return {"resultados_busca": bruto[:1500], "iteracoes": state["iteracoes"] + 1}

def node_avaliar_l(state: EstadoLoop) -> dict:
    q = (ChatPromptTemplate.from_template(
        "Pergunta: {p}\nResultado: {r}\nScore de suficiência 0.0-1.0:"
    ) | llm.with_structured_output(Qualidade)).invoke(
        {"p": state["mensagens"][0].content, "r": state["resultados_busca"]})
    return {"qualidade": q.score}

def node_responder_l(state: EstadoLoop) -> dict:
    resp = llm.invoke([HumanMessage(
        f"Pergunta: {state['mensagens'][0].content}\nFonte: {state['resultados_busca']}")])
    return {"mensagens": [resp]}

# 👉 LACUNA 3 e 4: os dois retornos da função de roteamento
def decidir_continuar_l(state: EstadoLoop) -> str:
    if state["qualidade"] >= THRESHOLD or state["iteracoes"] >= MAX_ITER:
        return ___    # qualidade suficiente → responder
    return ___        # senão → volta ao loop

builder_loop = StateGraph(EstadoLoop)
builder_loop.add_node("buscar", node_buscar_l)
builder_loop.add_node("avaliar", node_avaliar_l)
builder_loop.add_node("responder", node_responder_l)
builder_loop.add_edge(START, "buscar")
builder_loop.add_edge("buscar", "avaliar")
builder_loop.add_conditional_edges("avaliar", decidir_continuar_l,
                                   {"buscar": "buscar", "responder": "responder"})
builder_loop.add_edge("responder", END)
pesquisador_l = builder_loop.compile(checkpointer=MemorySaver())

PERGUNTAS = ["O que é grounding em sistemas RAG?", "Como funciona o checkpointing do LangGraph?"]
for pergunta in PERGUNTAS:
    config_l = {"configurable": {"thread_id": f"fix13-{pergunta[:12]}"}}
    pesquisador_l.invoke({"mensagens": [HumanMessage(pergunta)],
                          "resultados_busca": "", "qualidade": 0.0, "iteracoes": 0}, config=config_l)
    v = pesquisador_l.get_state(config_l).values
    esgotou = " · esgotou MAX" if v["iteracoes"] >= MAX_ITER else ""
    print(f"THRESHOLD={THRESHOLD} | iteracoes={v['iteracoes']} | score={v['qualidade']:.2f}{esgotou} | {pergunta[:40]}")


### Exercício 4 — Fonte do domínio no node buscar · ★★☆ · 10 min

*Individual · Colab*

1. Complete o `node_buscar_ex`: a query lida do estado e o join do conteúdo dos documentos.
2. Registre os 3 nodes no grafo — o `buscar` com a fonte nova, `avaliar` e `responder` prontos do Exercício 3.
3. Complete a função e a aresta condicional de volta do loop, compile com checkpointer e renderize o grafo.
4. Rode com 1 pergunta do domínio — a aresta `avaliar → buscar` deve aparecer no diagrama.

> **💡 Dica:** a única linha que muda no `node_buscar` é a fonte — `run(query)` vira `retriever.invoke(query)`. `avaliar` e `decidir_continuar` são agnósticos à fonte.


In [ ]:
# 👉 LACUNA: o pesquisador com a fonte do CKP02 (requer o Exercício 3 executado)
# Se /content/ckp02 não existir ainda, o Chroma cria a coleção vazia — o loop roda até o MAX.
!pip install langchain-chroma -q
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from IPython.display import Image, display

embeddings_ex = OllamaEmbeddings(model="nomic-embed-text")
retriever_ex = Chroma(persist_directory="/content/ckp02",
                      embedding_function=embeddings_ex).as_retriever(search_kwargs={"k": 3})

# 👉 LACUNA 1: query do estado e extração do conteúdo de cada documento
def node_buscar_ex(state: EstadoLoop) -> dict:
    docs = retriever_ex.invoke(___)              # query = pergunta do estado
    bruto = "\n".join(___ for d in docs)         # conteúdo de cada doc
    return {"resultados_busca": bruto[:1500],
            "iteracoes": state["iteracoes"] + 1}

builder_ex4 = StateGraph(EstadoLoop)
# 👉 LACUNA 2: registre o node buscar (fonte nova)
builder_ex4.add_node("buscar", ___)
builder_ex4.add_node("avaliar", node_avaliar_l)
builder_ex4.add_node("responder", node_responder_l)
builder_ex4.add_edge(START, "buscar")
builder_ex4.add_edge("buscar", "avaliar")
# 👉 LACUNA 3: função de roteamento na aresta condicional de volta
builder_ex4.add_conditional_edges("avaliar", ___,
                                  {"buscar": "buscar", "responder": "responder"})
builder_ex4.add_edge("responder", END)
pesquisador_ex4 = builder_ex4.compile(checkpointer=MemorySaver())

try:
    display(Image(pesquisador_ex4.get_graph().draw_mermaid_png()))
except Exception:
    print(pesquisador_ex4.get_graph().draw_mermaid())

config_ex4 = {"configurable": {"thread_id": "fix13-ckp02"}}
pesquisador_ex4.invoke({"mensagens": [HumanMessage("O que é grounding em RAG?")],
                        "resultados_busca": "", "qualidade": 0.0, "iteracoes": 0}, config=config_ex4)
print("Iteracoes:", pesquisador_ex4.get_state(config_ex4).values["iteracoes"])


## 📚 Referências da aula

- Docs LangGraph — Guia completo: StateGraph, checkpointing, HITL. langchain-ai.github.io/langgraph/tutorials/introduction
- Docs LangGraph HITL — interrupt_before, update_state, invoke(None). langchain-ai.github.io/langgraph/concepts/human_in_the_loop
- Blog Anthropic Engineering — "Building Effective Agents" (2025). Secao sobre checkpointing e revisao humana. anthropic.com/engineering/building-effective-agents
- Tool Mermaid Live Editor — Para visualizar o output de draw_mermaid() sem instalar playwright. mermaid.live
- Livro Russell, S.; Norvig, P. — Inteligencia Artificial. 3ª ed. Pearson, 2016. Cap. 3 — Resolucao de problemas como busca: base conceitual dos grafos de estado no LangGraph.
- Livro Bornet, P.; Wirtz, J. et al. — Agentic Artificial Intelligence. World Scientific, 2025. A analogia do "funcionário recém-contratado" e HITL como fase de confiança, não trava permanente — a fundamentação por trás do interrupt_before desta aula.
- Livro Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 4 — Reflection: o modelo Producer-Critic que justifica separar os nós buscar e avaliar no grafo pesquisador desta aula.

---

**Proxima Aula — Aula 14 (ultima)** — Spec-Driven Development e Encerramento
  
Retrospectiva do semestre · Spec-Driven Development · LangSmith · Proximos passos na carreira · Encerramento.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*